In [ ]:
# ==========================================
# Locate / upload ECGData.mat
# ==========================================
import os

MAT_PATH = "ECGData.mat"
if not os.path.exists(MAT_PATH):
    try:                                  # Colab: prompt for an upload
        from google.colab import files
        uploaded = files.upload()
        MAT_PATH = next(iter(uploaded))
    except ImportError:                   # local: file must sit next to the notebook
        raise FileNotFoundError(f"{MAT_PATH} not found - put it next to this notebook.")

# ==========================================
# Load MAT file
# ==========================================
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt

mat = sio.loadmat(MAT_PATH)

print(mat.keys())

: 

In [ ]:
# ==========================================
# Extract ECGData Structure
# ==========================================

ECGData = mat['ECGData']

# MATLAB struct fields
Data = ECGData['Data'][0, 0]
Labels = ECGData['Labels'][0, 0]

print("Data Shape   :", Data.shape)
print("Labels Shape :", Labels.shape)

# Convert MATLAB cell/string labels to clean Python strings
# np.array (not a list): the selection code below needs boolean masking
label_list = np.array([str(Labels[i, 0][0]) for i in range(len(Labels))])

print("Unique Labels:", np.unique(label_list))


In [ ]:
# ==========================================
# Segment ECG into 500 sample windows
# ==========================================

SEG_LEN = 500

segments = []
segment_labels = []

for i in range(Data.shape[0]):

    signal = Data[i, :]
    label = label_list[i]

    n_segments = len(signal) // SEG_LEN

    for j in range(n_segments):

        start = j * SEG_LEN
        end = start + SEG_LEN

        seg = signal[start:end]

        segments.append(seg)
        segment_labels.append(label)

segments = np.array(segments)
segment_labels = np.array(segment_labels)

print("Segments Shape :", segments.shape)
print("Labels Shape   :", segment_labels.shape)

In [ ]:
classes = np.unique(segment_labels)

print("Available Classes:", classes)

plt.figure(figsize=(15,10))

for idx, cls in enumerate(classes):

    indices = np.where(segment_labels == cls)[0]

    print(cls, ":", len(indices))

    if len(indices) == 0:
        continue

    sample_idx = indices[0]

    plt.subplot(len(classes),1,idx+1)
    plt.plot(segments[sample_idx])
    plt.title(f'{cls} ECG Segment')
    plt.grid(True)

plt.tight_layout()
plt.show()

## Variational Mode Decomposition (VMD)

VMD is applied to each complete ECG recording before the 500-sample segmentation step.
The decomposition returns `K` band-limited modes whose sum reconstructs the ECG signal.

Every record in `ECGData.mat` is sampled at **128 Hz**, so centre frequencies are
reported in Hz (`omega * FS`) rather than normalised units.

**Two caveats on this design.**

1. *Cost.* VMD on a full 65 536-sample record takes ~30 s, so all 162 records is
   ~80 minutes single-core. Decomposing 500-sample windows instead is far cheaper
   and lets the mode set adapt to local rhythm.
2. *`alpha = 2000` is a poor fit for ECG.* It is the value carried around in most VMD
   tutorials, but it forces modes to be near-pure tones, which a QRS complex is not.
   On this data `alpha = 2000` leaves a residual of ‖r‖/‖x‖ ≈ 0.21 (~4.3 % of signal
   *energy*) and puts no mode above ~23 Hz; `alpha ≈ 5` leaves ≈ 0.003 with *better*
   mode separation. See `Arrythmia_v2.ipynb` §5.


In [ ]:
# ==========================================
# Install / import VMD
# ==========================================

# vmdpy provides the VMD implementation used below.
# It is also maintained within sktime, but vmdpy is the
# smallest dependency for this notebook.
%pip install -q vmdpy   # kernel-aware: installs into THIS kernel's env

from vmdpy import VMD
from tqdm.auto import tqdm

print("VMD imported successfully")


In [ ]:
# ==========================================
# VMD Parameters
# ==========================================

FS = 128.0             # Hz - every record in ECGData.mat is resampled to 128 Hz
VMD_ALPHA = 2000       # bandwidth constraint  (see note: alpha=5 fits ECG far better)
VMD_TAU = 0.0          # noise-tolerance / relaxed fidelity
VMD_K = 5              # number of modes
VMD_DC = 0             # do not force the first mode to DC
VMD_INIT = 1           # uniformly initialize center frequencies
VMD_TOL = 1e-7         # convergence tolerance

print("VMD configuration:")
print(f"  fs    = {FS} Hz")
print(f"  alpha = {VMD_ALPHA}")
print(f"  tau   = {VMD_TAU}")
print(f"  K     = {VMD_K}")
print(f"  DC    = {VMD_DC}")
print(f"  init  = {VMD_INIT}")
print(f"  tol   = {VMD_TOL}")


In [ ]:
# ==========================================
# Apply VMD to one complete ECG recording
# ==========================================

def decompose_ecg_vmd(signal, alpha=VMD_ALPHA, tau=VMD_TAU,
                       K=VMD_K, DC=VMD_DC, init=VMD_INIT,
                       tol=VMD_TOL):
    """Decompose one 1-D ECG signal into K variational modes."""
    signal = np.asarray(signal, dtype=np.float64).ravel()

    if signal.ndim != 1:
        raise ValueError("signal must be one-dimensional")
    if len(signal) < 10:
        raise ValueError("signal is too short for VMD")

    modes, mode_spectra, center_frequencies = VMD(
        signal, alpha, tau, K, DC, init, tol
    )

    return modes, mode_spectra, center_frequencies


# Use the first ARR recording as a concrete example.
example_index = 0
example_signal = Data[example_index, :]
example_label = label_list[example_index]

example_modes, example_mode_spectra, example_omega = decompose_ecg_vmd(example_signal)
example_reconstruction = np.sum(example_modes, axis=0)

print("Example recording:", example_index)
print("Label:", example_label)
print("Original shape:", example_signal.shape)
print("Modes shape:", example_modes.shape)
print("Center frequencies (Hz):", np.round(np.sort(example_omega[-1]) * FS, 3))
print(
    "Relative reconstruction error:",
    np.linalg.norm(example_signal - example_reconstruction)
    / np.linalg.norm(example_signal),
)


In [ ]:
# ==========================================
# Visualize the VMD decomposition
# ==========================================

fig, axes = plt.subplots(VMD_K + 2, 1, figsize=(15, 2.2 * (VMD_K + 2)), sharex=True)

axes[0].plot(example_signal, linewidth=1)
axes[0].set_title(f"Original ECG - {example_label}")
axes[0].grid(True)

for k in range(VMD_K):
    axes[k + 1].plot(example_modes[k], linewidth=1)
    axes[k + 1].set_title(f"VMD Mode {k + 1}")
    axes[k + 1].grid(True)

axes[-1].plot(example_signal - example_reconstruction, linewidth=1)
axes[-1].set_title("Reconstruction Residual")
axes[-1].set_xlabel("Sample")
axes[-1].grid(True)

plt.tight_layout()
plt.show()

# Optional: show the learned center frequencies in normalized units.
plt.figure(figsize=(8, 4))
plt.stem(np.arange(1, VMD_K + 1), example_omega[-1])
plt.xlabel("Mode")
plt.ylabel("Normalized center frequency")
plt.title("VMD Center Frequencies")
plt.grid(True)
plt.show()


In [ ]:
# ==========================================
# Apply VMD to the ECG dataset
# ==========================================

# ECGData.mat is stored SORTED BY CLASS (96 ARR, then 30 CHF, then 36 NSR), so
# taking the first N records yields an all-ARR set. Sample per class instead.
#
# Cost: VMD on one full 65536-sample record is ~30 s, so MAX_VMD_SIGNALS = None
# means roughly 80 minutes single-core. Keep it small unless you mean it.
MAX_VMD_SIGNALS = 9          # None -> every recording (~80 min)

if MAX_VMD_SIGNALS is None:
    sel_idx = np.arange(Data.shape[0])
else:
    _rng = np.random.default_rng(0)
    _cls = np.unique(label_list)
    _per = max(1, MAX_VMD_SIGNALS // len(_cls))
    sel_idx = np.sort(np.concatenate([
        _rng.choice(np.where(label_list == c)[0], min(_per, (label_list == c).sum()),
                    replace=False) for c in _cls]))

n_to_process = len(sel_idx)
print("selected records:", sel_idx)
print("class balance   :", dict(zip(*np.unique(label_list[sel_idx], return_counts=True))))

# Store results as a list so we do not allocate one large 4-D array.
# Each element has shape (VMD_K, 65536).
vmd_results = []
vmd_omegas = []
vmd_labels = []
vmd_record_ids = []

for i in tqdm(sel_idx, desc="VMD decomposition"):
    modes, _, omega = decompose_ecg_vmd(Data[i, :])
    vmd_results.append(modes.astype(np.float32))
    vmd_omegas.append(np.sort(omega[-1]).astype(np.float32))
    vmd_labels.append(label_list[i])
    vmd_record_ids.append(i)          # group key - needed to avoid patient leakage

vmd_omegas = np.asarray(vmd_omegas)
vmd_labels = np.asarray(vmd_labels)
vmd_record_ids = np.asarray(vmd_record_ids)

print(f"Processed {n_to_process} / {Data.shape[0]} recordings")
print("Each VMD result shape:", vmd_results[0].shape if vmd_results else None)
print("VMD labels shape:", vmd_labels.shape)
print("Center-frequency matrix shape:", vmd_omegas.shape)


In [ ]:
# ==========================================
# Convert VMD modes into 500-sample windows
# ==========================================

# One SAMPLE per (recording, window), with the K modes kept as CHANNELS.
#
# The earlier version emitted one row per (recording, mode, window) and gave every
# row the recording's label. That is wrong: it asserts that the baseline-wander mode
# of an ARR record is itself "an ARR example", and it lets modes of the same window
# land on both sides of a train/test split. Modes are channels of one observation,
# not independent observations.

VMD_SEG_LEN = SEG_LEN

vmd_windows, vmd_window_labels, vmd_window_record = [], [], []

for pos, modes in enumerate(vmd_results):
    n_segments = modes.shape[1] // VMD_SEG_LEN
    for j in range(n_segments):
        sl = slice(j * VMD_SEG_LEN, (j + 1) * VMD_SEG_LEN)
        vmd_windows.append(modes[:, sl])              # (K, SEG_LEN)
        vmd_window_labels.append(vmd_labels[pos])
        vmd_window_record.append(vmd_record_ids[pos])

vmd_windows       = np.asarray(vmd_windows, dtype=np.float32)   # (n_windows, K, SEG_LEN)
vmd_window_labels = np.asarray(vmd_window_labels)
vmd_window_record = np.asarray(vmd_window_record)

print("VMD windows shape :", vmd_windows.shape, " (n_windows, K, SEG_LEN)")
print("Labels shape      :", vmd_window_labels.shape)
print("Record ids shape  :", vmd_window_record.shape, "-> group key for a leak-free split")
print("class balance     :", dict(zip(*np.unique(vmd_window_labels, return_counts=True))))


In [ ]:
# ==========================================
# Example: inspect one 500-sample VMD window
# ==========================================

record_to_plot = 0
window_to_plot = 0

start = window_to_plot * VMD_SEG_LEN
end = start + VMD_SEG_LEN

fig, axes = plt.subplots(VMD_K, 1, figsize=(14, 2 * VMD_K), sharex=True)
if VMD_K == 1:
    axes = [axes]

for k, ax in enumerate(axes):
    ax.plot(vmd_results[record_to_plot][k, start:end])
    ax.set_ylabel(f"Mode {k + 1}")
    ax.grid(True)

axes[-1].set_xlabel("Sample within 500-sample window")
fig.suptitle(
    f"VMD Modes | Record {record_to_plot} | Label {vmd_labels[record_to_plot]}",
    y=1.02
)
plt.tight_layout()
plt.show()
